# Semana 2 — Tarea del proyecto ancla · *Soft sensor* de % de sílice
### Cuaderno guiado (ALUMNO)

En el cuaderno satélite aprendiste la regresión lineal con datos sintéticos. Ahora la aplicas a la planta de flotación: estimar el **% de sílice del concentrado** a partir de las variables de proceso, con la tabla horaria de los ~6 meses (`flotacion_semisintetica_horaria.csv`).

**Sobre los datos: versión semisintética.** Desde esta semana trabajamos con una versión *semisintética* del dataset. Las variables de proceso (alimentación, reactivos, caudal, pH, densidad, aire y nivel) son **las reales** de la planta que exploraste en la Semana 1: mismas horas, mismos valores y el mismo hueco de registros. Lo que cambia es la **salida**: la sílice del concentrado (y con ella `fuera_spec`) se ha **simulado** con un modelo del proceso que no se publica. La usamos porque nos permite aplicar mejor los conceptos del curso: sabemos que hay una relación que aprender, así que podrás ver con claridad qué aporta cada técnica y comprobar qué métodos capturan mejor esa relación. Tu trabajo es el mismo que con datos reales: descubrir esa relación a partir de los datos, sin conocer el modelo que los generó.

**Qué te damos hecho.** El dataset ya lo exploraste en la Semana 1. La sección 1 está resuelta: carga la tabla, fija las entradas y la salida y resume lo que conviene recordar de los datos (rangos, evolución de la sílice y el hueco de registros). Solo tienes que ejecutarla y leerla.

**Qué entregas esta semana (tarea semanal)**

- Las **secciones 2 y 3**: una **regresión lineal** evaluada con una **partición temporal** y comparada con el **baseline** (predecir siempre la media). Completa las celdas `# TODO` y responde a las dos preguntas en las celdas «Tu respuesta».
- Se entrega como archivo `.ipynb` en el Campus Virtual, ejecutado de principio a fin sin errores.
- Si ya hiciste la tarea semanal con el cuaderno anterior, sobre el dato real, puedes entregar ese. La sección 4 hazla con este cuaderno.

**Qué no entregas esta semana.** La sección 4 es tu parte del **proyecto final** correspondiente a esta semana: una pregunta clave, una ampliación fundamental y dos ampliaciones más. No hay que hacerla ahora: se entrega con el proyecto final.

**Cómo trabajar**

- Ejecuta las celdas **en orden**, de arriba abajo.
- En las celdas con `# TODO` hay huecos marcados con `None` o con un comentario. Complétalos siguiendo la **pista** del comentario: casi siempre es una sola línea.
- Después de cada paso importante hay una celda **Comprueba**. Si da error, revisa ese paso antes de seguir: los pasos siguientes dependen de él.
- Un resultado modesto **no es un fracaso**: lo importante es que el método sea correcto y que interpretes el resultado con honestidad.

## 0. Preparación

Cargamos las librerías. Ejecuta la celda sin cambiar nada.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

pd.set_option('display.width', 120)

## 1. Los datos (resuelto)

Esta sección ya está hecha: ejecuta sus celdas sin cambiar nada y lee los resultados. Las secciones siguientes usan las variables que se definen aquí: `tabla`, `n_filas`, `objetivo` y `variables_proceso`.

### 1a. Cargar la tabla

La tabla tiene **una fila por hora**. Es la tabla horaria que aprendiste a construir en la Semana 1, ahora con todo el histórico (marzo–septiembre de 2017) y con la sílice semisintética. Cada fila trae la media horaria de las variables de proceso y el valor de sílice de esa hora.

| Columna | Qué contiene | Papel en la tarea |
|---|---|---|
| `date` | hora a la que corresponde la fila | metadato |
| `n_muestras` | número de registros de 20 s promediados en esa hora | metadato |
| `silice` | % de sílice en el concentrado (semisintética: hace el papel del análisis de laboratorio) | **salida (lo que predecimos)** |
| `iron_feed`, `silica_feed` | % de hierro y de sílice del mineral que entra | entrada |
| `almidon_medio` | caudal de almidón | entrada |
| `amina_media` | caudal de amina | entrada |
| `caudal_pulpa` | caudal de pulpa | entrada |
| `ph_medio` | pH de la pulpa | entrada |
| `densidad` | densidad de la pulpa | entrada |
| `aire_medio` | aire inyectado, media de las 7 columnas de flotación | entrada |
| `nivel_medio` | nivel de espuma, media de las 7 columnas | entrada |
| `fuera_spec` | 1 si la sílice de esa hora supera el umbral de especificación | derivada de la salida |

Los datos están en la carpeta `datasets`, al lado de la carpeta de esta semana. Si la celda siguiente da `FileNotFoundError`, revisa que has descargado el CSV en `SII/datasets/`.

In [ ]:
ruta_datos = '../datasets/flotacion_semisintetica_horaria.csv'
tabla = pd.read_csv(ruta_datos, parse_dates=['date'])

n_filas = tabla.shape[0]
n_columnas = tabla.shape[1]

print("Filas:", n_filas, "| columnas:", n_columnas)
tabla.head()

Comprobamos que has cargado el fichero correcto:

In [ ]:
assert n_filas == 4097, "Deberías tener 4097 filas: ¿has cargado flotacion_semisintetica_horaria.csv?"
assert n_columnas == 13, "Deberías tener 13 columnas"
print("Correcto")

Antes de modelar, comprobamos si faltan valores (celdas vacías, `NaN`):

In [ ]:
valores_ausentes = tabla.isna().sum()
print("Valores ausentes por columna:")
print(valores_ausentes)

### 1b. Entradas y salida (sin fugas)

Un *soft sensor* debe estimar la sílice **antes** de que llegue el resultado del laboratorio. Por eso, como entradas solo valen las variables que la planta mide en tiempo real: las de papel «entrada» en la tabla de 1a.

Quedan **prohibidas** como entrada:

- `silice`: es lo que queremos predecir.
- `fuera_spec`: se calcula a partir de la sílice. Usarla sería darle al modelo la respuesta (**fuga de datos**).
- `date` y `n_muestras`: son metadatos, no describen el proceso.

> Regla práctica: si un modelo «acierta» demasiado, sospecha de una fuga.

In [ ]:
objetivo = 'silice'

variables_proceso = ['iron_feed', 'silica_feed', 'almidon_medio', 'amina_media',
                     'caudal_pulpa', 'ph_medio', 'densidad', 'aire_medio', 'nivel_medio']

print(len(variables_proceso), "entradas:", variables_proceso)

### 1c. Primer vistazo

Un recordatorio de lo que viste en la Semana 1, ahora con todo el histórico.

`describe()` da, para cada columna, el número de valores, la media, la desviación típica (`std`), el mínimo, los cuartiles y el máximo.

In [ ]:
resumen = tabla.describe()
resumen = resumen.round(2)
resumen

La sílice a lo largo del tiempo:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(tabla['date'], tabla[objetivo], lw=0.6)
ax.set_ylabel("% sílice")
ax.set_title("% de sílice a lo largo de los ~6 meses")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

En la gráfica de la sílice se ve un tramo recto al principio: la planta estuvo unos días sin registros. Medimos cuántas horas pasan entre filas consecutivas:

In [ ]:
salto_entre_filas = tabla['date'].diff()
segundos_entre_filas = salto_entre_filas.dt.total_seconds()
horas_entre_filas = segundos_entre_filas / 3600

hueco_mayor = horas_entre_filas.max()
fila_hueco = horas_entre_filas.idxmax()
fecha_fin_hueco = tabla.loc[fila_hueco, 'date']
print(f"Hueco mayor: {hueco_mayor:.0f} horas ({hueco_mayor / 24:.1f} días), termina en la fila {fila_hueco}: {fecha_fin_hueco}")

**Lo que conviene recordar de los datos**

- No faltan valores en ninguna columna.
- La sílice se mueve entre 0,6 % y 5,7 %, con una media de 2,51 %.
- Las entradas tienen escalas muy distintas: el almidón ronda los 2900 y el pH, los 10. Por eso estandarizaremos las entradas en la sección 3.
- Hay un hueco de 319 horas (13,3 días) sin registros, que termina el 29 de marzo de 2017. En la tarea no hay que hacer nada con él. Importará en las ampliaciones (sección 4), si construyes variables que miran horas anteriores.

## 2. Dividir en train y test respetando el tiempo

Entrenamos con las **primeras** horas (80 %) y evaluamos con las **últimas** (20 %). Así simulamos el uso real: el modelo se construye con el pasado y se pone a prueba con un periodo que no ha visto.

¿Por qué no repartir las horas al azar, como en el satélite? Dos horas seguidas de la planta son casi iguales. Si se mezclan al azar, cada hora del test tiene a su vecina en el train y el modelo parece mejor de lo que es. En el satélite los datos no tenían orden temporal; aquí sí. Si quieres comprobarlo, tienes una ampliación en la sección 4.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `n_train` | número de horas del train (80 % de `n_filas`) |
| `train`, `test` | primeras `n_train` filas de `tabla` y el resto |
| `X_train_sin_escalar`, `X_test_sin_escalar` | las 9 entradas de proceso de cada parte, en sus unidades originales |
| `y_train`, `y_test` | % de sílice real de cada parte |

In [ ]:
esta_ordenada = tabla['date'].is_monotonic_increasing
print("¿Filas en orden cronológico?", esta_ordenada)

fraccion_train = 0.8
n_train = int(n_filas * fraccion_train)

# === TODO ===
# train = las primeras n_train filas;  test = el resto.
# pista: tabla.iloc[:n_train]   y   tabla.iloc[n_train:]
train = None   # TODO
test = None    # TODO

inicio_train = train['date'].min()
fin_train = train['date'].max()
inicio_test = test['date'].min()
fin_test = test['date'].max()
print(f"train: {len(train)} horas  ({inicio_train:%Y-%m-%d} -> {fin_train:%Y-%m-%d})")
print(f"test : {len(test)} horas  ({inicio_test:%Y-%m-%d} -> {fin_test:%Y-%m-%d})")

**Comprueba:**

In [ ]:
assert len(train) == 3277 and len(test) == 820, "El train debe tener 3277 horas y el test 820"
assert fin_train < inicio_test, "Todas las horas del train deben ser anteriores a las del test"
print("Correcto")

Así queda la división en el tiempo (ejecuta sin cambiar nada):

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(train['date'], train[objetivo], lw=0.6, label='train (80 % inicial)')
ax.plot(test['date'], test[objetivo], lw=0.6, label='test (20 % final)')
ax.set_ylabel("% sílice")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Separamos entradas (`X`) y salida (`y`) en cada parte:

In [ ]:
X_train_sin_escalar = train[variables_proceso]
X_test_sin_escalar = test[variables_proceso]

# === TODO ===
# Guarda la salida (la columna objetivo) de cada parte.
# pista: y_train = train[objetivo]
y_train = None   # TODO
y_test = None    # TODO

print("X_train:", X_train_sin_escalar.shape, "| y_train:", y_train.shape)
print("X_test :", X_test_sin_escalar.shape, "| y_test :", y_test.shape)

## 3. Regresión lineal frente al baseline

### 3a. Baseline: predecir siempre la media

Antes de cualquier modelo necesitamos una **vara de medir**: el error que se comete con la regla más tonta posible, predecir para cada hora del test la media de la sílice del train. Un modelo solo vale la pena si mejora este **baseline**.

Usamos tres métricas, todas calculadas sobre el **test**:

| Métrica | Qué mide | Unidades | Mejor si… |
|---|---|---|---|
| MAE | error absoluto medio: cuánto te equivocas de media | % de sílice | más pequeño |
| RMSE | raíz del error cuadrático medio: castiga más los errores grandes | % de sílice | más pequeño |
| R² | fracción de la variación de la sílice que explica el modelo | sin unidades | más cerca de 1 |

Sobre el R²: **1** es predicción perfecta; **0** equivale a predecir siempre la media; **negativo** significa que el modelo lo hace *peor* que predecir la media.

In [ ]:
media_train = y_train.mean()
print(f"Media de la sílice en el train: {media_train:.3f} %")

# El baseline predice el mismo valor (la media del train) para cada hora del test
n_test = len(y_test)
pred_baseline = np.full(n_test, media_train)

# === TODO ===
# Calcula el MAE y el RMSE del baseline sobre el test.
# pista: mean_absolute_error(y_test, pred_baseline)   y   root_mean_squared_error(y_test, pred_baseline)
mae_baseline = None    # TODO
rmse_baseline = None   # TODO

print(f"BASELINE   MAE = {mae_baseline:.3f}   RMSE = {rmse_baseline:.3f}")

**Comprueba:** deberías obtener un MAE del baseline de aproximadamente 0,74.

In [ ]:
assert abs(mae_baseline - 0.743) < 0.005, "Revisa el baseline: usa la media del TRAIN y evalúa sobre el TEST"
print("Correcto")

### 3b. Regresión lineal

El modelo es el mismo del satélite, ahora con 9 entradas:

$$\widehat{\text{sílice}} = a_0 + a_1\,\texttt{iron\_feed} + a_2\,\texttt{silica\_feed} + \dots + a_9\,\texttt{nivel\_medio}$$

`LinearRegression` de scikit-learn calcula los coeficientes que minimizan el error cuadrático medio, como la ecuación normal del satélite. No hace falta programar el descenso de gradiente.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `escalador` | `StandardScaler` ajustado **solo** con `X_train_sin_escalar` |
| `X_train`, `X_test` | entradas estandarizadas (media 0 y desviación 1, calculadas con el train) |
| `modelo` | la regresión lineal entrenada |
| `pred_train`, `pred_test` | predicciones del modelo en cada parte |
| `mae_test`, `rmse_test`, `r2_test` | métricas del modelo en el test |
| `r2_train` | R² del modelo en el propio train |

**Primero, estandarizar las entradas.** Para las predicciones de `LinearRegression` no es imprescindible, pero es la práctica habitual: lo necesitarás para comparar coeficientes, para la regularización y para modelos basados en distancias (todo ello en las ampliaciones).

El escalador se ajusta **solo con el train**. El test representa el futuro: no podemos usar su media ni su desviación para preparar el modelo.

In [ ]:
escalador = StandardScaler()

# === TODO ===
# Ajusta el escalador SOLO con las entradas del train (una línea).
# pista: escalador.fit(X_train_sin_escalar)


X_train = escalador.transform(X_train_sin_escalar)
X_test = escalador.transform(X_test_sin_escalar)

medias_train = X_train.mean(axis=0)
medias_test = X_test.mean(axis=0)
print("Medias de las entradas estandarizadas en el train:", medias_train.round(2))
print("Medias de las entradas estandarizadas en el test: ", medias_test.round(2))

En el train las medias salen 0, por construcción (`-0.` también es 0: es un redondeo). En el test **no** salen exactamente 0, y es lo esperable: el periodo de test tiene sus propias condiciones de operación. Si te salen 0 en ambos, has ajustado el escalador con el test.

### 3c. Entrenar y predecir

In [ ]:
modelo = LinearRegression()

# === TODO ===
# 1) Entrena el modelo con X_train e y_train (una línea).   pista: modelo.fit(X_train, y_train)
# 2) Predice el test.                                        pista: pred_test = modelo.predict(X_test)


pred_test = None   # TODO

pred_train = modelo.predict(X_train)
print("Primeras 5 predicciones del test:", pred_test[:5].round(2))

### 3d. Métricas: ¿bate al baseline?

In [ ]:
# === TODO ===
# Calcula las tres métricas del modelo en el TEST (compara y_test con pred_test).
# pista: mean_absolute_error(...), root_mean_squared_error(...), r2_score(...)
mae_test = None    # TODO
rmse_test = None   # TODO
r2_test = None     # TODO

r2_train = r2_score(y_train, pred_train)

print(f"{'':12s} {'MAE':>6s} {'RMSE':>6s} {'R²':>7s}")
print(f"{'baseline':12s} {mae_baseline:6.3f} {rmse_baseline:6.3f} {0:+7.3f}")
print(f"{'lineal':12s} {mae_test:6.3f} {rmse_test:6.3f} {r2_test:+7.3f}")
print(f"\nR² del lineal en el train: {r2_train:+.3f}   |   en el test: {r2_test:+.3f}")

**Comprueba:** deberías obtener, en el test, un MAE de aproximadamente 0,63 y un R² de aproximadamente 0,22.

In [ ]:
assert abs(mae_test - 0.632) < 0.005, "Revisa la sección 3: el MAE del test debería ser aproximadamente 0,63"
assert abs(r2_test - 0.225) < 0.005, "Revisa la sección 3: el R² del test debería ser aproximadamente 0,22"
print("Correcto")

**Pregunta 3.1.** ¿Bate tu modelo al baseline? ¿Por cuánto, en % de sílice (diferencia de MAE)? Explica con palabras qué significa tu R² del test.

**Tu respuesta:** *escribe aquí*

### 3e. Real frente a predicho

Ejecuta sin cambiar nada. Arriba, la sílice real y la predicha a lo largo del test. Abajo, cada hora como un punto: si el modelo fuera perfecto, todos los puntos caerían sobre la diagonal.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(test['date'], y_test, lw=0.6, label='real')
ax.plot(test['date'], pred_test, lw=0.8, label='predicho (lineal)')
ax.axhline(media_train, color='gray', ls='--', lw=0.8, label='baseline (media del train)')
ax.set_ylabel("% sílice")
ax.set_title("Test: sílice real y predicha")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

valor_minimo = y_test.min()
valor_maximo = y_test.max()
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(y_test, pred_test, s=4, alpha=0.4)
ax.plot([valor_minimo, valor_maximo], [valor_minimo, valor_maximo], 'k--', lw=0.8)
ax.set_xlabel("sílice real (%)")
ax.set_ylabel("sílice predicha (%)")
ax.set_title("Test: predicho frente a real")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Pregunta 3.2.** Mirando las dos gráficas, ¿en qué situaciones falla más el modelo? Fíjate en los valores altos y bajos de la sílice y en si la predicción sigue los cambios rápidos. Con todo ello, ¿sirve este modelo para estimar la sílice en la planta? ¿Qué ventaja da frente a no tener modelo y qué limitaciones tiene?

**Tu respuesta:** *escribe aquí*

> **Antes de entregar:** reinicia el kernel y ejecuta todo (en VS Code, «Restart» y luego «Run All»; en Colab, «Entorno de ejecución → Reiniciar y ejecutar todo»). Comprueba que no hay errores y que todas las celdas «Comprueba» dicen «Correcto». Guarda y sube el `.ipynb` al Campus Virtual.

## 4. Proyecto final

Esta sección **no se entrega esta semana**. Es tu parte del **proyecto final** correspondiente a la Semana 2: puedes hacerla durante el curso o en el mes de trabajo autónomo, y se entrega con el proyecto final.

**Pregunta clave de la semana:** *¿Hasta qué punto puede fiarse el operador de la sílice que estima tu modelo en horas que no ha visto?*

En la tarea semanal dimos por buena la partición temporal. En la comparación (4.2) la pondrás a prueba frente a una partición al azar, que trata las horas como si fueran independientes.

La sección tiene tres partes, y cada una se apoya en la anterior: la ampliación fundamental (4.1), la comparación de técnicas (4.2) y una técnica novedosa (4.3). Respalda cada afirmación con un experimento: lo importante no es ganar, sino explicar por qué un modelo funciona mejor o peor.

**Reglas para todo lo que hagas**

- Las de la tarea semanal: como entradas, solo variables de proceso; el escalador, y cualquier otro ajuste, solo con los datos con los que entrenas; evaluación final en el test, frente al baseline.
- Cualquier elección (qué entradas usar, valores de hiperparámetros, el número de vecinos…) se hace sobre una **muestra de validación**, **sin mirar el test**: en la evaluación temporal, con la validación de 4.1.

### 4.1 Ampliación fundamental (para aprobado)

Primero ejecuta la validación temporal, que te damos resuelta; después, haz el trabajo de la fundamental.

#### Validación temporal (resuelto)

Para elegir entre varias opciones necesitas medir el error en horas que el modelo no haya visto y que **no** sean las del test. Si eliges con el test, su cifra deja de ser una estimación honesta del error en horas nuevas. Por eso reservamos las **últimas horas del train** como validación, respetando el orden temporal: es el hold-out de la sección 4 del satélite, pero sin barajar.

La forma de trabajar es:

1. Entrena cada opción con `train_ajuste` (el escalador también se ajusta solo con `train_ajuste`) y mide su error en `validacion`.
2. Elige la opción con mejor resultado en `validacion`.
3. Entrena la opción elegida con todo el `train` y evalúala **una sola vez** en el `test`.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `n_validacion` | número de horas de validación (20 % del train) |
| `n_ajuste` | número de horas de `train_ajuste` |
| `train_ajuste` | primeras horas del train: para entrenar mientras comparas opciones |
| `validacion` | últimas horas del train: para elegir sin mirar el test |

Ejecuta la celda sin cambiar nada.

In [ ]:
fraccion_validacion = 0.2
n_validacion = int(len(train) * fraccion_validacion)
n_ajuste = len(train) - n_validacion

train_ajuste = train.iloc[:n_ajuste]
validacion = train.iloc[n_ajuste:]

inicio_validacion = validacion['date'].min()
fin_validacion = validacion['date'].max()
print(f"train_ajuste: {len(train_ajuste)} horas")
print(f"validacion  : {len(validacion)} horas  ({inicio_validacion:%Y-%m-%d} -> {fin_validacion:%Y-%m-%d})")
print(f"test        : {len(test)} horas")

#### Lo que tienes que hacer

Con la partición temporal de la tarea semanal:

1. ¿Tu modelo de la tarea semanal **se subentrena o se sobreentrena**? Compara el R² del train y el del test (apartado 3d) y justifica tu respuesta.
2. Haz una **ingeniería de características básica**: añade a las entradas sus **cuadrados y sus productos** dos a dos, como en la sección 3 del satélite. `PolynomialFeatures(degree=2, include_bias=False)`, de `sklearn.preprocessing`, los genera en una línea. Aplícalo a las entradas ya estandarizadas, como en la tarea semanal: con las unidades originales, las columnas nuevas tendrían escalas muy distintas (el almidón al cuadrado ronda los 9 millones y la densidad al cuadrado, 3), y el ajuste saldría algo peor por errores de redondeo. Compara varias opciones (por ejemplo, las 9 entradas, con cuadrados, con cuadrados y productos o con un grado mayor) con la validación temporal de arriba, elige una y pruébala una sola vez en el test.
3. ¿Mejora frente al modelo de la tarea semanal y frente al baseline? ¿Se subentrena o se sobreentrena ahora?
4. Conclusión, en pocas líneas: ¿qué error le comunicarías al operador y cuánto puede fiarse de la estimación en horas que el modelo no ha visto?

**Opciones propias.** Puedes añadir otras opciones que no estén en el enunciado, siempre que respeten las reglas; justifica por qué las pruebas.

Añade las celdas de código que necesites.

In [ ]:
# Tu código para 4.1


**Tus respuestas (4.1):**

1. *escribe aquí*
2. *escribe aquí*
3. *escribe aquí*
4. *escribe aquí*

### 4.2 Comparación de técnicas (para notable)

- **Partición al azar.** Evalúa también el modelo más prometedor en 4.1 tratando las horas como independientes, de dos maneras:
  - con una partición al azar del 80/20 (`train_test_split(tabla, test_size=0.2, random_state=0)`, de `sklearn.model_selection`);
  - con validación cruzada *K*-fold barajada sobre toda la tabla (`KFold(n_splits=5, shuffle=True, random_state=0)`), como en la sección 4 del satélite.

  ¿Qué cifra da cada forma de evaluar? ¿Elegirías el mismo modelo? ¿Cuál imita lo que le pasará al modelo en la planta?
- **Ridge y lasso** (`Ridge` y `Lasso`, de `sklearn.linear_model`; λ se llama `alpha`). Ridge está en la sección 5 del satélite. **Busca una forma de elegir λ sin mirar el test y justifícala.** Estandariza antes todas las entradas, también las nuevas.
- **Tabla enriquecida (opcional).** `flotacion_semisintetica_horaria_enriquecida.csv` trae el aire y el nivel de cada columna de flotación por separado y las desviaciones típicas dentro de cada hora.

Para cada opción, di si se subentrena o se sobreentrena y explica por qué funciona mejor o peor; por ejemplo, qué entradas conserva el lasso.

**Opciones propias.** Puedes añadir otras opciones que no estén en el enunciado, siempre que respeten las reglas; justifica por qué las pruebas.

In [ ]:
# Tu código para 4.2


**Tu respuesta (4.2):** *escribe aquí*

### 4.3 Técnica novedosa (para sobresaliente)

Aplica otros modelos de regresión, por ejemplo los k vecinos (`KNeighborsRegressor`, de `sklearn.neighbors`), u otros que encuentres o que veamos más adelante en el curso. Compáralos con tus mejores modelos anteriores, con la evaluación temporal y con la partición al azar de 4.2, y explica por qué funcionan mejor o peor.

**Opciones propias.** Puedes añadir otras opciones que no estén en el enunciado, siempre que respeten las reglas; justifica por qué las pruebas.

In [ ]:
# Tu código para 4.3


**Tu respuesta (4.3):** *escribe aquí*